# Gravimetric modeling — reproducible manuscript figures

**Scientific status:** controlled **synthetic numerical experiment**, not observed gravity from Marajó. This notebook supersedes previous versions and fixes inconsistent source geometry, ignored station elevations, extrapolation artifacts, and repeated refinement calculations.

**Outputs:** one file per panel: `figure1a`–`figure1d`, `figure2a`–`figure2c`, `figure3a`–`figure3c`, `figure4a`–`figure4f`, `figure5a`–`figure5e`, and supplemental `figureS1a`–`figureS1c`. All PNG/PDF/TIFF plus numerical CSV/NPZ and a methodology manifest are saved in `Figures/` and `outputs/reproducible_figures/`.

**Scientific method:** The common geological model uses the original package's **topography (elevation up)** and **Moho (depth down)** surfaces, interpolated **bilinearly** on their existing 11 × 11 grids. The source occupies the finite horizontal footprint and the vertical interval from `top=-topographic elevation` to `bottom=Moho depth`. Cartesian coordinates and observations are z-positive-down. Density is a stated, spatially constant contrast (320 kg/m³). A **continuous-surface numerical reference** integrates the vertical attraction analytically in z and by composite Gauss–Legendre quadrature in x and y. Three geometries are compared against that same physical source. The benchmark compares their actual source-cell approximations with the common reference and does not claim that any geometry is universally superior.

The reference is a **converged numerical quadrature approximation, not an exact analytical field**. The quadrature-convergence error is explicitly measured; full publication settings should only be accepted after numerical tolerance checks. The synthetic anomaly `B` is *constructed* as the reference regional field plus two stated local rectangular anomalies, so `R_ref = B-G_ref` is physically consistent. It is **not** the unrelated supplied synthetic Bouguer CSV. No field-data claims are made.

**Before running:** extract the complete revised ZIP (the notebook, `gravimetria_marajo.py`, and `examples/` are included). Install `numpy scipy shapely pandas matplotlib cmcrameri jupyterlab nbformat` (also install `ipykernel`). Set `FULL_PUBLICATION_RUN = False` for a lower-cost diagnostic; set `True` for the high-resolution 21-realization experiment. The notebook keeps persistent caches to avoid repeated long computations.

**Important:** Figure 5c now shows the absolute error distribution for each geometry, and Figures 5d and 5e show *separate finite-difference forward responses* for Moho depth and density contrast; mesh errors belong in Figure 3 and are not treated as physical-property sensitivity. The original `.tex` captions and numerical tables describe a different earlier experiment. **They must be updated using this notebook's actual outputs**, especially the definition of the reference, observation elevations, density contrasts, errors, and experiment size. Figures should not be represented as validated until all checks pass.

**Verification improvements (November 2026 review):** The full-run reference refinement is checked against a strict 0.00005 mGal maximum-difference threshold at 121 validation stations, including domain boundaries. Meshes are checked for complete, non-overlapping coverage. Figure 3b timings are freshly measured in the current session, rather than imported from an earlier machine's cache. Runtime measurements are system-dependent and are not comparable across different hardware.


In [1]:
from pathlib import Path
import sys, json, hashlib, time, warnings, platform
import numpy as np
import pandas as pd
import scipy
from scipy.interpolate import RegularGridInterpolator
import matplotlib.pyplot as plt
from matplotlib.collections import PolyCollection
from shapely.geometry import Polygon, box

ROOT=Path.cwd().resolve()
if ROOT.name=='notebooks' and not (ROOT/'gravimetria_marajo.py').is_file(): ROOT=ROOT.parent
if not (ROOT/'gravimetria_marajo.py').is_file():
    raise FileNotFoundError('Place prismas.ipynb next to gravimetria_marajo.py in the extracted prismas.zip root')
sys.path.insert(0,str(ROOT))
import gravimetria_marajo as gm

# Parameters for full computation and quick numerical diagnostic.
FULL_PUBLICATION_RUN = True  # Final mode: 101x101 nodes and a 21-member ensemble. Set False only for diagnostics.
STRICT_CRAMERI = True         # Install cmcrameri; fallback is never mislabeled as Crameri.
SEED=431
PLOT_N = 101 if FULL_PUBLICATION_RUN else 51
ENSEMBLE_PER_GEOMETRY = 7 if FULL_PUBLICATION_RUN else 2
REGIONAL_TILE_M = 250.0 if FULL_PUBLICATION_RUN else 500.0
REF_QUADRATURE_ORDER = 2 if FULL_PUBLICATION_RUN else 2
DPI=400
RHO=320.0  # kg/m³ density contrast
OBS_Z=-500.0  # m, same observation elevation convention as the input synthetic CSV
BBOX=(0.,10000.,0.,10000.)

try:
    from cmcrameri import cm as cmc
    GRAV_CMAP=cmc.roma_r
    GRAV_CMAP_NAME='Crameri roma_r'
except ImportError:
    if STRICT_CRAMERI:
        raise ImportError('Install scientific colormaps first: python -m pip install cmcrameri')
    warnings.warn('cmcrameri absent; using matplotlib cividis as explicitly labeled preview ONLY')
    GRAV_CMAP=plt.get_cmap('cividis')
    GRAV_CMAP_NAME='matplotlib cividis (diagnostic fallback)'
ERR_CMAP='coolwarm'
FIG=ROOT/'Figures'; OUT=ROOT/'outputs'/'reproducible_figures'; CACHE=OUT/'cache'
for p in (FIG,OUT,CACHE):p.mkdir(parents=True,exist_ok=True)
plt.rcParams.update({'font.size':10,'axes.labelsize':10,'xtick.labelsize':9,'ytick.labelsize':9,
                     'axes.spines.top':False,'axes.spines.right':False,'pdf.fonttype':42,
                     'figure.dpi':115,'savefig.dpi':DPI})

MODEL_PARAMETERS=dict(full_run=FULL_PUBLICATION_RUN,plot_n=PLOT_N,seed=SEED,
                      rho_kg_m3=RHO,obs_z_m=OBS_Z,regional_tile_m=REGIONAL_TILE_M,
                      reference_quadrature_order=REF_QUADRATURE_ORDER,
                      ensemble_per_geometry=ENSEMBLE_PER_GEOMETRY)
print('Gravity module:',gm.__file__)
print('Colormap:',GRAV_CMAP_NAME)
print('Parameters:',MODEL_PARAMETERS)


Gravity module: C:\Users\Usuario\Python_Notebooks\prismas\gravimetria_marajo.py
Colormap: Crameri roma_r
Parameters: {'full_run': True, 'plot_n': 101, 'seed': 431, 'rho_kg_m3': 320.0, 'obs_z_m': -500.0, 'regional_tile_m': 250.0, 'reference_quadrature_order': 2, 'ensemble_per_geometry': 7}


In [2]:
# Functions for exported graphics and deterministic computation.

def digest(*args):
    h=hashlib.sha256()
    for item in args:
        if isinstance(item,np.ndarray):
            h.update(str((item.shape,str(item.dtype))).encode());h.update(np.ascontiguousarray(item).tobytes())
        else:h.update(repr(item).encode())
    return h.hexdigest()[:24]

def cached_array(tag,func,*fingerprint):
    cache_key=digest(tag,MODEL_PARAMETERS,*fingerprint)
    path=CACHE/f'{cache_key}.npz'
    if path.exists():
        with np.load(path) as f: return f['data'].copy()
    t=time.perf_counter();res=np.asarray(func(),float)
    np.savez_compressed(path,data=res)
    print(f'Calculated {tag}: shape={res.shape}, elapsed={time.perf_counter()-t:.2f}s',flush=True)
    return res

def rms(a):
    arr=np.asarray(a,float)
    return float(np.sqrt(np.mean(arr*arr)))

def save(fig,stem):
    for ext in ('png','pdf','tif'):
        kwargs={'dpi':DPI} if ext!='pdf' else {}
        fig.savefig(FIG/f'{stem}.{ext}',bbox_inches='tight',facecolor='white',**kwargs)
    plt.close(fig)
    print('Exported',stem,flush=True)

def positive_or_signed_levels(a,signed=False,n=31):
    arr=np.asarray(a,float)
    assert np.isfinite(arr).all(), 'Nonfinite field; refusing to draw contours'
    if signed:
        bound=max(float(np.max(np.abs(arr))),1e-12)
        return np.linspace(-bound,bound,n)
    low,high=np.min(arr),np.max(arr)
    if high-low <1e-13: high=low+1e-13
    return np.linspace(low,high,n)

def map_panel(name,X,Y,data,label,gravity=True,limits=None,points=False):
    arr=np.asarray(data,float)
    assert X.shape==Y.shape==arr.shape
    signed=not gravity
    levels=np.linspace(*limits,31) if limits is not None else positive_or_signed_levels(arr,signed=signed)
    fig,ax=plt.subplots(figsize=(5.7,4.8),constrained_layout=True)
    c=ax.contourf(X/1000,Y/1000,arr,levels=levels,
                  cmap=GRAV_CMAP if gravity else ERR_CMAP,extend='neither')
    ax.set(xlabel='x (km)',ylabel='y (km)')
    ax.set_aspect('equal',adjustable='box')
    if points: ax.scatter(X[::max(1,len(X)//10),::max(1,len(X)//10)]/1000,
                          Y[::max(1,len(Y)//10),::max(1,len(Y)//10)]/1000,
                          s=2,c='#4c4c4c',alpha=.2)
    cb=fig.colorbar(c,ax=ax,pad=.025,shrink=.9,format='%.3g')
    cb.set_label(label)
    save(fig,name)

def xy_grid(n=PLOT_N):
    return np.meshgrid(np.linspace(BBOX[0],BBOX[1],n),np.linspace(BBOX[2],BBOX[3],n))

X,Y=xy_grid()
XF=X.ravel();YF=Y.ravel();ZF=np.full(len(XF),OBS_Z)
assert PLOT_N>=50


## 1. Original surface files and consistent physical source

In [3]:
# The two supplied surfaces are 11x11 tabulations at 0–10 km.
# Original topography is elevation UP, so the top is depth-down = -elevation.
# The Moho is supplied as depth DOWN; no fictitious ~2 km substitute is introduced.

def load_surface(path):
    a=pd.read_csv(path)
    for col in ('x_m','y_m','z_m'):
        if col not in a.columns:raise ValueError(f'Missing {col} in {path}')
    xs=np.sort(a.x_m.unique());ys=np.sort(a.y_m.unique())
    vals=a.pivot(index='y_m',columns='x_m',values='z_m').reindex(index=ys,columns=xs).to_numpy(float)
    if not np.isfinite(vals).all():raise ValueError('Incomplete input surface grid')
    return xs,ys,vals

topo_path=ROOT/'examples'/'synthetic_topography.csv'
moho_path=ROOT/'examples'/'synthetic_moho.csv'
xt,yt,topo_vals=load_surface(topo_path)
xm,ym,moho_vals=load_surface(moho_path)
assert np.array_equal(xt,xm) and np.array_equal(yt,ym)
TOPO_ELEVATION=RegularGridInterpolator((yt,xt),topo_vals,bounds_error=True)
MOHO_DEPTH=RegularGridInterpolator((ym,xm),moho_vals,bounds_error=True)

def top_depth(x,y):
    x,y=np.broadcast_arrays(np.asarray(x,float),np.asarray(y,float))
    return -TOPO_ELEVATION(np.column_stack((y.ravel(),x.ravel()))).reshape(x.shape)

def base_depth(x,y):
    x,y=np.broadcast_arrays(np.asarray(x,float),np.asarray(y,float))
    return MOHO_DEPTH(np.column_stack((y.ravel(),x.ravel()))).reshape(x.shape)

T=top_depth(X,Y);H=base_depth(X,Y)
assert np.min(T)>OBS_Z, 'Observation station must remain above all source material'
assert np.min(H-T)>0
# User-supplied anomaly is preserved as original material, NOT used in the new synthetic reference.
ORIGINAL_ANOMALY=pd.read_csv(ROOT/'examples'/'synthetic_observed_bouguer.csv')
print('Original synthetic anomaly stations:',len(ORIGINAL_ANOMALY), 'not reused as observations in the new experiment')
print('Top depth range (m):',T.min(),T.max())
print('Moho depth range (m):',H.min(),H.max())


Original synthetic anomaly stations: 49 not reused as observations in the new experiment
Top depth range (m): -113.878154 -0.0
Moho depth range (m): 28000.0 29187.883692


## 2. Independent continuous-surface numerical quadrature (vertical integral evaluated analytically)

In [4]:
# g_z = G * Delta_rho * integral_{footprint} [1/sqrt(R_xy²+(top-z_obs)²)
#                                        -1/sqrt(R_xy²+(base-z_obs)²)] dx dy
# with top/bottom sampled from the same *bilinearly interpolated* physical surfaces.
# The x-y integral is evaluated by composite Gauss–Legendre quadrature.
# Output units: mGal; positive downward.

def integrate_surface_vertical(xobs,yobs,zobs,rho=RHO,spacing=REGIONAL_TILE_M,order=2,
                               moho_shift=0.,top_shift=0.,chunk=256):
    xobs=np.asarray(xobs,float).ravel();yobs=np.asarray(yobs,float).ravel()
    zobs=np.broadcast_to(zobs,xobs.shape).astype(float)
    edges_x=np.linspace(BBOX[0],BBOX[1],int(np.ceil((BBOX[1]-BBOX[0])/spacing))+1)
    edges_y=np.linspace(BBOX[2],BBOX[3],int(np.ceil((BBOX[3]-BBOX[2])/spacing))+1)
    v,w=np.polynomial.legendre.leggauss(order)
    gx=[]; gy=[]; weights=[]
    for ix in range(len(edges_x)-1):
        px=(edges_x[ix]+edges_x[ix+1])/2+(edges_x[ix+1]-edges_x[ix])/2*v
        wx=(edges_x[ix+1]-edges_x[ix])/2*w
        for iy in range(len(edges_y)-1):
            py=(edges_y[iy]+edges_y[iy+1])/2+(edges_y[iy+1]-edges_y[iy])/2*v
            wy=(edges_y[iy+1]-edges_y[iy])/2*w
            xi,yi=np.meshgrid(px,py)
            gx.extend(xi.ravel());gy.extend(yi.ravel())
            weights.extend(np.outer(wy,wx).ravel())
    sx=np.asarray(gx);sy=np.asarray(gy);sw=np.asarray(weights)
    st=top_depth(sx,sy)+top_shift
    sb=base_depth(sx,sy)+moho_shift
    if np.min(st-zobs.max())<=0:raise ValueError('Observation lies at/below top of source')
    assert np.all(sb>st)
    out=np.zeros_like(xobs)
    for a in range(0,len(xobs),chunk):
        b=min(a+chunk,len(xobs))
        r2=(sx[None,:]-xobs[a:b,None])**2+(sy[None,:]-yobs[a:b,None])**2
        za=zobs[a:b,None]
        f=1/np.sqrt(r2+(st[None,:]-za)**2)-1/np.sqrt(r2+(sb[None,:]-za)**2)
        out[a:b]=gm.G*rho*gm.SI_TO_MGAL*(f@sw)
    return out

# Accuracy verification with independent refinement of quadrature tiles.
CHECK_N=11
XC,YC=np.meshgrid(np.linspace(BBOX[0],BBOX[1],CHECK_N),np.linspace(BBOX[2],BBOX[3],CHECK_N))
chkx=XC.ravel();chky=YC.ravel();chkz=np.full(len(chkx),OBS_Z)
q_spacings=([1000.,500.,250.,125.] if FULL_PUBLICATION_RUN else [1000.,500.,250.])
q_fields={}
for spacing in q_spacings:
    q_fields[spacing]=cached_array(('quadrature_check',spacing),
        lambda s=spacing:integrate_surface_vertical(chkx,chky,chkz,spacing=s,order=2),
        spacing,chkx,chky,chkz,topo_vals,moho_vals)
q_rows=[]
for a,b in zip(q_spacings[:-1],q_spacings[1:]):
    q_rows.append({'tile_coarse_m':a,'tile_fine_m':b,
                   'rmse_change_mGal':rms(q_fields[a]-q_fields[b]),
                   'max_abs_change_mGal':float(np.max(np.abs(q_fields[a]-q_fields[b])))})
q_df=pd.DataFrame(q_rows)
q_df.to_csv(OUT/'reference_quadrature_refinement.csv',index=False)
print(q_df.to_string(index=False))
# Tolerance is deliberately far below the minimum geometry RMSE seen in full runs.
# This tests convergence on the 11x11 station subset, not a certified global bound.
REF_QUALITY_TOL = 5e-5 if FULL_PUBLICATION_RUN else 5e-4   # mGal
REF_QUALITY_MEASURED = float(q_df.iloc[-1]['max_abs_change_mGal'])
if REF_QUALITY_MEASURED > REF_QUALITY_TOL:
    raise RuntimeError(
        f'Quadrature reference failed station-level refinement check: '
        f'{REF_QUALITY_MEASURED:.6g} > {REF_QUALITY_TOL:.6g} mGal. '
        'Refine the integration before interpreting geometry-error magnitudes.'
    )
print(f'Reference refinement passed: {REF_QUALITY_MEASURED:.7g} mGal '
      f'(limit {REF_QUALITY_TOL:.7g} mGal; {CHECK_N**2} stations).')


 tile_coarse_m  tile_fine_m  rmse_change_mGal  max_abs_change_mGal
        1000.0        500.0          0.096126             0.146025
         500.0        250.0          0.003611             0.007535
         250.0        125.0          0.000011             0.000035
Reference refinement passed: 3.452221e-05 mGal (limit 5e-05 mGal; 121 stations).


## 3. Forward modeling in the same physical volume with three planar discretizations

In [5]:
# Finite, clipped footprints. Top and bottom of each column sampled from the
# SAME bilinearly-interpolated physical surfaces. This is the only geometry-induced
# approximation being studied, so the comparisons have a clear interpretation.

def footprints(kind,spacing,realization=0):
    xmin,xmax,ymin,ymax=BBOX
    if kind=='quadrangular':
        sx=((realization*0.173)%1-.5)*spacing
        sy=((realization*0.317)%1-.5)*spacing
        boundary=box(xmin,ymin,xmax,ymax)
        polys=[]
        for x0 in np.arange(xmin-spacing+sx,xmax+spacing,spacing):
            for y0 in np.arange(ymin-spacing+sy,ymax+spacing,spacing):
                geom=box(x0,y0,x0+spacing,y0+spacing).intersection(boundary)
                if geom.area>1e-6:polys.append(np.asarray(geom.exterior.coords[:-1],float))
        return polys
    if kind=='hexagonal':
        rad=spacing/np.sqrt(3)
        dy=np.sqrt(3)*rad;dx=1.5*rad
        sx=((realization*.177)%1-.5)*dx
        sy=((realization*.233)%1-.5)*dy
        boundary=box(*[xmin,ymin,xmax,ymax])
        polys=[]
        for i in range(-3,int(np.ceil((xmax-xmin)/dx))+5):
            for j in range(-3,int(np.ceil((ymax-ymin)/dy))+5):
                cx=xmin+i*dx+sx;cy=ymin+j*dy+(i%2)*dy/2+sy
                inter=Polygon(gm.regular_polygon(cx,cy,rad,n=6)).intersection(boundary)
                if inter.geom_type=='Polygon' and inter.area>1e-6:
                    polys.append(np.asarray(inter.exterior.coords[:-1],float))
        return polys
    if kind=='voronoi':
        # Existing deterministic Voronoi routine clips to exact same footprint.
        return gm._voronoi_polygon_footprints(xmin,xmax,ymin,ymax,spacing,seed=SEED+realization*13)
    raise ValueError(kind)

def model_cells(polygons,rho=RHO,moho_shift=0.,top_shift=0.):
    c=np.array([[Polygon(p).centroid.x,Polygon(p).centroid.y] for p in polygons])
    top=top_depth(c[:,0],c[:,1])+top_shift
    bot=base_depth(c[:,0],c[:,1])+moho_shift
    assert np.all(bot>top)
    return [gm.Cell(np.asarray(p,float),float(t),float(b),rho)
            for p,t,b in zip(polygons,top,bot)]

def field_from_cells(x,y,z,cells):
    # Consistent vectorized observations; sum each source cell exactly with
    # the package's analytic polygon edge formula, no gridding/interpolation.
    return gm.gz_cells(x,y,z,cells,method='polygon')

def modeled(kind,spacing,realization,x,y,z,rho=RHO,moho_shift=0.):
    key=(kind,spacing,realization,rho,moho_shift,'gravity')
    return cached_array(key,lambda:field_from_cells(x,y,z,model_cells(
        footprints(kind,spacing,realization),rho=rho,moho_shift=moho_shift)),
        key,np.asarray(x),np.asarray(y),np.asarray(z),topo_vals,moho_vals)


# Mesh-integrity checks: all cell polygons should tile the common footprint.
# Neither gaps nor double-counted overlaps are permissible in these comparisons.
from shapely.ops import unary_union

def verify_footprint(kind, spacing, realization=0, tol=1e-6):
    polygons = footprints(kind, spacing, realization)
    shapes = [Polygon(p) for p in polygons]
    if not shapes or not all(p.is_valid and p.area > 0 for p in shapes):
        raise RuntimeError(f'Invalid mesh polygons: {kind} {spacing}m')
    domain = box(BBOX[0], BBOX[2], BBOX[1], BBOX[3])
    union = unary_union(shapes)
    coverage = union.symmetric_difference(domain).area / domain.area
    overlap = (sum(p.area for p in shapes) - union.area) / domain.area
    if coverage > tol or abs(overlap) > tol:
        raise RuntimeError(f'Mesh integrity failure for {kind}/{spacing}: '
                           f'coverage mismatch={coverage:.3g}, overlap={overlap:.3g}')
    return dict(geometry=kind, spacing_m=spacing, realization=realization,
                cells=len(polygons), coverage_mismatch=coverage, overlap_fraction=overlap)

MESH_VERIFICATION = pd.DataFrame([
    verify_footprint(k, s, 0)
    for s in [1750., 875., 437.5, 750.]
    for k in ['quadrangular', 'hexagonal', 'voronoi']
])
MESH_VERIFICATION.to_csv(OUT/'mesh_integrity_verification.csv', index=False)
print('Mesh integrity verification passed for all three geometries.')


Mesh integrity verification passed for all three geometries.


## Figure 1 — Responses of four different source geometries

In [6]:
# Separate, controlled geometry demonstration. These are not error maps.
# Same density contrast and approximately matched lateral areas / thicknesses.
# Cartesian and spherical geometry fields are demonstrations, NOT exact-equivalent sources.
grid1=np.linspace(-3500,3500,61 if FULL_PUBLICATION_RUN else 41)
A,B=np.meshgrid(grid1,grid1);Z=np.zeros_like(A)
rho_demo=350.
rect=(-750.,750.,-650.,650.,900.,2700.)
rv=np.array([[-750.,-650.],[750.,-650.],[750.,650.],[-750.,650.]])
area_rect=(rect[1]-rect[0])*(rect[3]-rect[2])
hex_R=np.sqrt(2*area_rect/(3*np.sqrt(3)))
poly=np.array([[-1.1,-.55],[-.25,-1.05],[.75,-.82],[1.2,-.06],[.64,.82],[-.3,1.1],[-1.05,.4]])
poly*=np.sqrt(area_rect/abs(gm.signed_polygon_area(poly)))

g_rect=gm.gz_rectangular(A,B,Z,rect,rho_demo)
g_hex=gm.gz_hexagonal(A,B,Z,0.,0.,hex_R,900.,2700.,rho_demo)
g_polygon=gm.gz_polygon(A,B,Z,poly,900.,2700.,rho_demo)
Re=6371000.;lat_w=(rect[3]-rect[2])/Re;lon_w=(rect[1]-rect[0])/Re
r1,r2=Re-2700,Re-900
g_tess=gm.gz_tesseroid(np.full_like(A,Re),B/Re,A/Re,r1,r2,
                       -lat_w/2,lat_w/2,-lon_w/2,lon_w/2,rho_demo,order=6)
fig1={'figure1a':g_rect,'figure1b':g_hex,'figure1c':g_polygon,'figure1d':g_tess}
lims=(float(min(np.min(g) for g in fig1.values())),float(max(np.max(g) for g in fig1.values())))
for tag,g in fig1.items():
    map_panel(tag,A,B,g,
              'Local inward radial gravity (mGal)' if tag=='figure1d' else '$g_z$ (mGal)',
              gravity=True,limits=lims)
np.savez_compressed(OUT/'figure1_fields.npz',x_m=A,y_m=B,**fig1)


Exported figure1a
Exported figure1b
Exported figure1c
Exported figure1d


## Figure 2 — Independent verification of the forward operators

In [7]:
P=gm.gz_polygon(A,B,Z,rv,900.,2700.,rho_demo)
Q=gm.gz_rectangular_quadrature(A,B,Z,rect,rho_demo,order=8)
map_panel('figure2a',A,B,(P-g_rect)*1e14,
          'Polygon minus rectangle ($10^{-14}$ mGal)',gravity=False)
map_panel('figure2b',A,B,(Q-g_rect)*1e8,
          'GLQ(8) minus rectangle ($10^{-8}$ mGal)',gravity=False)
ord=np.array([2,4,6,8,10,12,16],int)
xx,yy=np.meshgrid(np.linspace(-3000,3000,25),np.linspace(-3000,3000,25))
reference_exact=gm.gz_rectangular(xx,yy,0.,rect,rho_demo)
err=np.array([rms(gm.gz_rectangular_quadrature(xx,yy,0.,rect,rho_demo,order=int(j))-reference_exact) for j in ord])
fig,ax=plt.subplots(figsize=(5.3,4.2),constrained_layout=True)
ax.semilogy(ord,np.maximum(err,np.finfo(float).tiny),'o-',color='#2166ac')
ax.set(xlabel='Gauss–Legendre quadrature order',ylabel='RMSE (mGal)')
save(fig,'figure2c')
pd.DataFrame({'order':ord,'rmse_mGal':err}).to_csv(OUT/'figure2_operator_rmse.csv',index=False)


Exported figure2a
Exported figure2b
Exported figure2c


## Shared reference fields — no interpolation and no duplicated calculations

In [8]:
# Compute at every mapped station (actual 51x51 or 101x101 observations).
GREF=cached_array('continuous_reference_dense',lambda:integrate_surface_vertical(
    XF,YF,ZF,spacing=REGIONAL_TILE_M,order=REF_QUADRATURE_ORDER),
    XF,YF,ZF,topo_vals,moho_vals).reshape(X.shape)
# Independent shallow residual sources with explicitly defined parameters.
LOCAL_SOURCES=[(-1900.,-500.,-1900.,-450.,300.,1700.,450.),
               (1100.,2500.,600.,2000.,550.,1600.,-280.)]
def shallow_sources(x,y,z):
    field=np.zeros(np.shape(x),float)
    for x1,x2,y1,y2,zt,zb,rho in LOCAL_SOURCES:
        # shift local bodies into the 10km x 10km domain
        field+=gm.gz_rectangular(x,y,z,(x1+5000,x2+5000,y1+5000,y2+5000,zt,zb),rho)
    return field

RTRUE=shallow_sources(X,Y,np.full(X.shape,OBS_Z))
SYNTHETIC_B=GREF+RTRUE
assert np.allclose(SYNTHETIC_B-GREF,RTRUE)

# Geometry effects are differences relative to this common reference.
# The 875 m mesh retains interpretable geometric effects while reducing strong
# aliasing signatures of the earlier 1750 m comparison. No spatial smoothing is used.
KINDS=['quadrangular','hexagonal','voronoi']
SPACING_MAIN=875.  # Moderately refined comparison; coarse-mesh benchmark remains in Figure 3
MODELED_FIELDS={}
for kind in KINDS:
    MODELED_FIELDS[kind]=modeled(kind,SPACING_MAIN,0,XF,YF,ZF).reshape(X.shape)
    print('Main geometry field:',kind,'RMSE (mGal)=',rms(MODELED_FIELDS[kind]-GREF),flush=True)


Main geometry field: quadrangular RMSE (mGal)= 0.0023333231570431437
Main geometry field: hexagonal RMSE (mGal)= 0.0017389731070519055
Main geometry field: voronoi RMSE (mGal)= 0.0033896331654258423


## Figure 3 — Convergence and computational efficiency (no redundant high-resolution reference)

In [9]:
# Use the SAME independently computed continuous numerical reference for all geometries.
# Timings are measurements of actual computations; cached results are not benchmark timings.
xs=np.linspace(400,9600,13);ys=np.linspace(400,9600,13)
SX,SY=np.meshgrid(xs,ys);sx=SX.ravel();sy=SY.ravel();sz=np.full(len(sx),OBS_Z)
GCONT_SMALL=cached_array('continuous_reference_stations',lambda:integrate_surface_vertical(
    sx,sy,sz,spacing=REGIONAL_TILE_M,order=2),sx,sy,sz,topo_vals,moho_vals)
mesh_levels=[1750.,1125.,875.,625.,437.5] if FULL_PUBLICATION_RUN else [1750.,875.,437.5]
nrep=3 if FULL_PUBLICATION_RUN else 1
records=[];refinement_fields={}
for spacing in mesh_levels:
    print('Convergence spacing',spacing,'m',flush=True)
    for kind in KINDS:
        for run in range(nrep):
            polygons=footprints(kind,spacing,run)
            cells=model_cells(polygons)
            # Timings are measured NOW on the active CPU, irrespective of saved fields.
            # Never reuse time measurements from a different hardware/software session.
            # Median of three runs reduces the impact of short-lived scheduling noise.
            key=('benchmark',kind,spacing,run)
            tag=digest(key,MODEL_PARAMETERS,sx,sy,sz,topo_vals,moho_vals)
            path=CACHE/f'{tag}.npz'
            times=[]
            for _ in range(3):
                t0=time.perf_counter()
                fresh=field_from_cells(sx,sy,sz,cells)
                times.append(time.perf_counter()-t0)
            result=fresh
            runtime=float(np.median(times))
            np.savez_compressed(path,data=result,runtime=runtime)
            if kind=='quadrangular' and run==0: refinement_fields[spacing]=(result,len(polygons))
            records.append(dict(geometry=kind,spacing_m=spacing,realization=run,
                                n_cells=len(polygons),runtime_s=runtime,
                                rmse_mGal=rms(result-GCONT_SMALL)))
benchmark=pd.DataFrame(records)
benchmark.to_csv(OUT/'figure3_benchmark.csv',index=False)

sty={'quadrangular':('o','#2166ac'),'hexagonal':('s','#d6604d'),'voronoi':('^','#2a788e')}
fig,ax=plt.subplots(figsize=(5.4,4.3),constrained_layout=True)
for kind in KINDS:
    data=benchmark[benchmark.geometry==kind]
    mark,col=sty[kind]
    ax.scatter(data.n_cells,data.rmse_mGal,marker=mark,color=col,alpha=.35,s=18)
    med=data.groupby('spacing_m',as_index=False)[['n_cells','rmse_mGal']].median().sort_values('n_cells')
    ax.plot(med.n_cells,med.rmse_mGal,marker=mark,color=col,lw=1.5,label=kind.capitalize())
ax.set(xlabel='Number of cells',ylabel='RMSE relative to continuous-surface quadrature (mGal)',xscale='log',yscale='log')
ax.legend(frameon=False)
save(fig,'figure3a')

fig,ax=plt.subplots(figsize=(5.4,4.3),constrained_layout=True)
for kind in KINDS:
    a=benchmark[(benchmark.geometry==kind)&(benchmark.realization==0)].sort_values('n_cells')
    mark,col=sty[kind]
    ax.plot(a.runtime_s,a.rmse_mGal,'-',marker=mark,color=col,lw=1.5,label=kind.capitalize())
ax.set(xlabel='Forward runtime (s)',ylabel='RMSE relative to continuous-surface quadrature (mGal)',xscale='log',yscale='log')
ax.legend(frameon=False)
save(fig,'figure3b')

# Reference refinement: compare quadrature fields against successive smaller tiles.
# This is distinct from geometry-mesh refinement and is labeled accordingly.
fig,ax=plt.subplots(figsize=(5.4,4.3),constrained_layout=True)
a=q_df.copy()
ax.loglog(a.tile_fine_m,a.rmse_change_mGal,'o-',color='#2166ac')
ax.set(xlabel='Quadrature tile size (m)',ylabel='Successive quadrature RMSE (mGal)')
ax.invert_xaxis()
save(fig,'figure3c')


Convergence spacing 1750.0 m
Convergence spacing 1125.0 m
Convergence spacing 875.0 m
Convergence spacing 625.0 m
Convergence spacing 437.5 m
Exported figure3a
Exported figure3b
Exported figure3c


## Figure 4 — Physically consistent regional and residual maps (direct calculation, no extrapolation)

In [10]:
# All data exist at each plotted grid node: NO griddata, imshow or nearest fill.
# A common color scale is essential when comparing the input field and its reference.
common_g_limits=(float(min(SYNTHETIC_B.min(),GREF.min())),
                 float(max(SYNTHETIC_B.max(),GREF.max())))
map_panel('figure4a',X,Y,SYNTHETIC_B,'Synthetic gravity anomaly $B$ (mGal)',limits=common_g_limits)
map_panel('figure4b',X,Y,GREF,'Regional gravity $G_{ref}$ (mGal)',limits=common_g_limits)
# The signed residual contains known added local sources, NOT arbitrary noise.
residual_bound=float(np.max(np.abs(RTRUE)))
map_panel('figure4c',X,Y,RTRUE,'Known local-source residual $R_{ref}$ (mGal)',
          gravity=False,limits=(-residual_bound,residual_bound))

DELTA={k:GREF-MODELED_FIELDS[k] for k in KINDS}
shared_error_limit=max(np.max(np.abs(v)) for v in DELTA.values())
shared_error_limits=(-shared_error_limit,shared_error_limit)
for tag,kind in [('figure4d','quadrangular'),('figure4e','hexagonal'),('figure4f','voronoi')]:
    map_panel(tag,X,Y,DELTA[kind],
              '$\\Delta R=G_{ref}-G_m$ (mGal)',gravity=False,limits=shared_error_limits)
np.savez_compressed(OUT/'figure4_fields.npz',x_m=X,y_m=Y,B=SYNTHETIC_B,
                    Gref=GREF,Rref=RTRUE,delta_quadrangular=DELTA['quadrangular'],
                    delta_hexagonal=DELTA['hexagonal'],delta_voronoi=DELTA['voronoi'])
assert all(np.isfinite(v).all() for v in DELTA.values())


Exported figure4a
Exported figure4b
Exported figure4c
Exported figure4d
Exported figure4e
Exported figure4f


## Figure 5 — Discretization ensemble and parameter sensitivity

In [11]:
# 21 ensemble members for full mode (7 per geometry), 6 for preview (2 per geometry).
# Each member field is evaluated at all physical observation nodes.
ENSEMBLE=[]; ensemble_labels=[]
for kind in KINDS:
    for realization in range(ENSEMBLE_PER_GEOMETRY):
        field=modeled(kind,SPACING_MAIN,realization,XF,YF,ZF).reshape(X.shape)
        ENSEMBLE.append(GREF-field)
        ensemble_labels.append((kind,realization))
ENS=np.stack(ENSEMBLE)
assert len(ENS)==3*ENSEMBLE_PER_GEOMETRY
DR=np.std(ENS,axis=0,ddof=1)
RMS_STATION=np.sqrt(np.mean(ENS**2,axis=0))
map_panel('figure5a',X,Y,DR,'Discretization dispersion $D_R$ (mGal)')
map_panel('figure5b',X,Y,RMS_STATION,'Station-wise RMS of $\\Delta R_m$ (mGal)')

# Figure 5c: show each realization's whole-map RMSE by geometry.
# This resolves the overplotted 10k-point scatter and is more relevant for
# comparing discretizations; M=21 realizations total in publication mode.
error_rows=[]
for (kind,run),arr in zip(ensemble_labels,ENS):
    error_rows.append({'geometry':kind,'realization':run,
                       'map_rmse_mGal':rms(arr),
                       'map_mae_mGal':float(np.mean(np.abs(arr))),
                       'map_bias_mGal':float(np.mean(arr))})
error_df=pd.DataFrame(error_rows)
error_df.to_csv(OUT/'figure5c_ensemble_error_summary.csv',index=False)
fig,ax=plt.subplots(figsize=(5.5,4.3),constrained_layout=True)
labels=['Quadrangular','Hexagonal','Voronoi']
colors=['#28669a','#cb7655','#248777']
for j,kind in enumerate(KINDS):
    data=error_df.query('geometry == @kind')['map_rmse_mGal'].to_numpy()
    # Plot exact individual ensemble errors and median. Never fabricate a distribution.
    positions=j+0.08*np.linspace(-1,1,len(data))
    ax.scatter(positions,data,s=42,color=colors[j],alpha=.75,edgecolor='white',linewidth=.4,zorder=3)
    med=float(np.median(data))
    ax.hlines(med,j-.20,j+.20,colors=colors[j],linewidth=2.5,zorder=4)
    ax.annotate(f'{med:.4f}',(j,med),xytext=(0,9),textcoords='offset points',
                ha='center',fontsize=8,color=colors[j])
ax.set_xticks(range(3),labels)
ax.set_ylabel('Whole-map RMSE relative to reference (mGal)')
ax.set_ylim(bottom=0)
ax.grid(axis='y',alpha=.18)
save(fig,'figure5c')

# Physical-parameter sensitivities MUST be shown separately.
# Figure 5d: +/-400 m shift in the Moho, with all other properties fixed.
# Figure 5e: 250 and 450 kg/m^3 density contrasts, baseline 320 kg/m^3.
# No arbitrary normalizing scales, and no mesh-error bars mixed with physical changes.
G0=GCONT_SMALL
moho_rows=[dict(shift_m=0.0,rms_change_mGal=0.0)]
for shift in (-400.,400.):
    perturb=cached_array(('moho_shift',shift),
        lambda s=shift:integrate_surface_vertical(sx,sy,sz,spacing=REGIONAL_TILE_M,moho_shift=s),
        sx,sy,sz,shift,topo_vals,moho_vals)
    moho_rows.append(dict(shift_m=shift,rms_change_mGal=rms(perturb-G0)))
moho_df=pd.DataFrame(moho_rows).sort_values('shift_m')
moho_df.to_csv(OUT/'figure5d_moho_sensitivity.csv',index=False)
fig,ax=plt.subplots(figsize=(5.3,4.2),constrained_layout=True)
ax.plot(moho_df.shift_m,moho_df.rms_change_mGal,'s-',color='#2166ac',lw=1.7,ms=7)
ax.set(xlabel='Moho depth perturbation (m)',
       ylabel='RMS gravity response (mGal)')
ax.set_ylim(bottom=0);ax.axvline(0,color='0.65',lw=1,ls='--')
ax.grid(alpha=.18)
save(fig,'figure5d')

rho_rows=[dict(density_kg_m3=RHO,rms_change_mGal=0.0)]
for dens in (250.,450.):
    rho_rows.append(dict(density_kg_m3=dens,
                         rms_change_mGal=rms((dens/RHO-1)*G0)))
rho_df=pd.DataFrame(rho_rows).sort_values('density_kg_m3')
rho_df.to_csv(OUT/'figure5e_density_sensitivity.csv',index=False)
fig,ax=plt.subplots(figsize=(5.3,4.2),constrained_layout=True)
ax.plot(rho_df.density_kg_m3,rho_df.rms_change_mGal,'o-',color='#be6147',lw=1.7,ms=7)
ax.set(xlabel='Density contrast (kg m$^{-3}$)',
       ylabel='RMS gravity response (mGal)')
ax.set_ylim(bottom=0);ax.axvline(RHO,color='0.65',lw=1,ls='--')
ax.grid(alpha=.18)
save(fig,'figure5e')
np.savez_compressed(OUT/'figure5_ensemble.npz',ensemble=ENS,DR=DR,rms_station=RMS_STATION)
print('Ensemble count:',len(ENS))


Exported figure5a
Exported figure5b
Exported figure5c
Exported figure5d
Exported figure5e
Ensemble count: 21


## Supplementary Figure S1 — Footprint geometry and actual sampled Moho depths

In [12]:
# A true polygon-mesh map uses PolyCollection, not contourf over polygon centroids.
# This is the appropriate geometry visualization; only scalar-field maps use contourf.
for tag,kind in [('figureS1a','quadrangular'),('figureS1b','hexagonal'),('figureS1c','voronoi')]:
    poly=footprints(kind,750,0)
    centers=np.array([[Polygon(v).centroid.x,Polygon(v).centroid.y] for v in poly])
    vals=base_depth(centers[:,0],centers[:,1])/1000
    fig,ax=plt.subplots(figsize=(5.2,4.5),constrained_layout=True)
    collection=PolyCollection([np.asarray(v)/1000 for v in poly],array=np.asarray(vals),
                              cmap=GRAV_CMAP,edgecolor='#343434',lw=.3)
    ax.add_collection(collection)
    collection.set_clim(27.,30.)
    ax.scatter(centers[:,0]/1000,centers[:,1]/1000,s=.5,c='black',alpha=.4)
    ax.set(xlim=(0,10),ylim=(0,10),xlabel='x (km)',ylabel='y (km)')
    ax.set_aspect('equal')
    cb=fig.colorbar(collection,ax=ax,pad=.02,shrink=.92);cb.set_label('Moho depth (km)')
    save(fig,tag)


Exported figureS1a
Exported figureS1b
Exported figureS1c


## Quality checks, limitations, and export audit

In [13]:
# Never allow final output to silently assert that it reproduces the old manuscript tables.
if FULL_PUBLICATION_RUN and len(ENS)!=21:
    raise RuntimeError('Full ensemble must contain exactly 21 realizations')
if not np.allclose(SYNTHETIC_B-GREF,RTRUE,rtol=1e-12,atol=1e-10):
    raise RuntimeError('B - Gref does not equal the prescribed physical residual')
if not np.isfinite(np.stack([GREF,SYNTHETIC_B,RTRUE,DR,RMS_STATION])).all():
    raise RuntimeError('Nonfinite map values')
expected=[*(f'figure1{l}' for l in 'abcd'),*(f'figure2{l}' for l in 'abc'),
          *(f'figure3{l}' for l in 'abc'),*(f'figure4{l}' for l in 'abcdef'),
          *(f'figure5{l}' for l in 'abcde'),*(f'figureS1{l}' for l in 'abc')]
missing=[name for name in expected if not (FIG/f'{name}.png').is_file()]
if missing:raise RuntimeError(f'Missing figure outputs: {missing}')
manifest={
  'experiment':'Finite-footprint synthetic regional gravity using bilinear package surfaces',
  'not_real_data':True,'b_origin':'Gref plus two stated synthetic local bodies',
  'reference':'Independent composite x-y Gauss Legendre / analytic z integration',
  'original_bouguer_file_not_used':True,
  'continuous_reference_is_numerical_approximation':True,
  'source_top_depth':'negative of tabulated elevation-up in synthetic_topography.csv',
  'source_base_depth':'tabulated depth-down in synthetic_moho.csv',
  'station_z_m':OBS_Z,
  'map_grid_shape':list(X.shape),
  'model_parameters':MODEL_PARAMETERS,
  'scientific_colormap':GRAV_CMAP_NAME,
  'ensemble_member_count':len(ENS),
  'local_residual_prisms':LOCAL_SOURCES,
  'comparison_spacing_m':SPACING_MAIN,
  'meaning_figure5c':'distribution of whole-map RMSE across geometry realizations',
  'meaning_figure5d':'RMS gravity response to Moho depth perturbation in metres',
  'meaning_figure5e':'RMS gravity response to density contrast in kg/m^3',
  'quadrature_refinement':q_df.to_dict(orient='records'),
  'quadrature_verification_tolerance_mGal':REF_QUALITY_TOL,
  'quadrature_verification_measured_mGal':REF_QUALITY_MEASURED,
  'quadrature_verification_station_count':CHECK_N**2,
  'mesh_integrity_verification':MESH_VERIFICATION.to_dict(orient='records'),
  'benchmark_timing':'fresh median of 3 forward evaluations per case on the active machine',
  'figure_files':expected,
  'notebook_limitations':[
    'Quadrature was checked at 121 validation stations, not mathematically bounded at every point',
    'Reference quad convergence should be reassessed after changing source geometry',
    'Synthetic source is finite horizontally, not a global Moho model',
    'No field observations are analyzed and no geological interpretation is inferred',
    'Figures 5d and 5e are separate finite-difference parameter studies, not uncertainty distributions',
    'The provided manuscript captions and tables must be revised to reflect this experiment',
    'DPI measures export resolution; observation resolution is PLOT_N squared'
  ]
}
(OUT/'experiment_manifest.json').write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding='utf-8')
print(f'Exported {len(expected)} independently saved PNG/PDF/TIFF panels')
print('Output directory:',FIG)
print('Numerical data directory:',OUT)
print('Quadrature refinement maximum difference:',float(q_df.iloc[-1].max_abs_change_mGal),'mGal')
print('Full paper run:',FULL_PUBLICATION_RUN)


Exported 24 independently saved PNG/PDF/TIFF panels
Output directory: C:\Users\Usuario\Python_Notebooks\prismas\Figures
Numerical data directory: C:\Users\Usuario\Python_Notebooks\prismas\outputs\reproducible_figures
Quadrature refinement maximum difference: 3.452220735766787e-05 mGal
Full paper run: True
